# Deutsch-Jozsa Algorithm

Determine whether a function f : {0,1}ⁿ → {0,1} is **constant** or **balanced** in a single query. This example uses n = 2.

In [ ]:
import pennylane as qml
import numpy as np

## Oracles

- **Constant**: f(x) = 0 or f(x) = 1
- **Balanced**: f(x) = x₀ or f(x) = ¬x₀

In [ ]:
N = 2
dev = qml.device("default.qubit", wires=N + 1)

def oracle_constant_zero():
    pass

def oracle_constant_one():
    qml.PauliX(wires=N)

def oracle_balanced_identity():
    qml.CNOT(wires=[0, N])

def oracle_balanced_not():
    qml.PauliX(wires=0)
    qml.CNOT(wires=[0, N])
    qml.PauliX(wires=0)

## Deutsch-Jozsa circuit

If all input qubits measure |0⟩ after the query, f is constant; otherwise balanced.

In [ ]:
@qml.qnode(dev)
def deutsch_jozsa_circuit(oracle_fn):
    qml.PauliX(wires=N)
    qml.Hadamard(wires=range(N + 1))
    oracle_fn()
    qml.Hadamard(wires=range(N))
    return qml.probs(wires=range(N))

oracles = [
    ("f(x) = 0 (constant)", oracle_constant_zero),
    ("f(x) = 1 (constant)", oracle_constant_one),
    ("f(x) = x\u2080 (balanced)", oracle_balanced_identity),
    ("f(x) = \u00acx\u2080 (balanced)", oracle_balanced_not),
]

for name, oracle_fn in oracles:
    probs = deutsch_jozsa_circuit(oracle_fn)
    result = "constant" if np.isclose(probs[0], 1.0) else "balanced"
    measured = np.argmax(probs)
    measured_bits = format(measured, f"0{N}b")
    print(f"  Oracle: {name}")
    print(f"    Measured: |{measured_bits}\u27e9  \u2192  {result}")
    print()